# Multi-Agent Safety: Control Evals, Collusion, and Infection

**Goal**: After this session, you will be able to implement the core quantitative machinery of AI-control safety analysis from memory: audit-budget thresholding, the catch-vs-ship safety metric, signal-based monitor collusion, a paraphrasing defense, and infection propagation across an agent network.
**Time**: 1 hour.

## What and Why

AI control (Greenblatt & Shlegeris et al., arXiv:2312.06942) asks: *if the AIs in a deployment were actively trying to sabotage you, would your oversight catch them?* Its key open problem is **collusion** — an untrusted monitor deliberately letting the untrusted policy's attacks through (Shlegeris 2024; UK AISI Alignment Project 2025 lists this as a priority problem with £15M+ behind it). Multi-agent systems add a second bespoke failure: **injection propagation** — one compromised agent infecting the rest (Prompt Infection, arXiv:2410.07283: a 50-agent society saturates in <11 communication steps). This session builds toy-but-structurally-faithful versions of all of it.

## Key Formula

$$\text{safety} = P\big(\min\{t : \text{caught}_t\} < \min\{t : \text{shipped}_t\}\big)$$

**Where:**
- $t$ indexes tasks in a deployment episode; the untrusted policy backdoors each task with probability $p_{\text{attack}}$.
- $\text{caught}_t$: task $t$ is backdoored **and** its suspicion score exceeds the audit threshold (a human audit catches it; you win — shutdown).
- $\text{shipped}_t$: task $t$ is backdoored and **not** audited (the backdoor deploys; you lose).
- The audit threshold is set so only an `AUDIT_BUDGET` fraction of *clean* tasks get flagged (humans are expensive).

## Component Breakdown

- **Part 1 (recall)**: `audit_threshold` — turn an audit budget into a score threshold via a quantile of clean scores.
- **Part 2**: `episode_safety` — Monte Carlo the catch-vs-ship race; the safety number control evals report.
- **Part 3**: `collusion_safety` — a colluding monitor suppresses suspicion when it sees a coordination signal; watch safety collapse.
- **Part 4**: paraphrase defense — erase the signal channel with probability $q$; measure the safety-recovery curve.
- **Part 5**: `steps_to_saturation` — prompt-infection spread over chain / star / random agent graphs.

## References

- AI Control: Improving Safety Despite Intentional Subversion — arXiv:2312.06942 (ICML 2024)
- Buck Shlegeris, "How to prevent collusion when using untrusted models to monitor each other" (Sept 2024)
- Research Areas in AI Control (UK AISI Alignment Project, Aug 2025 — Stastny, Korbak, Shlegeris, ...)
- "When can we trust untrusted monitoring?" — arXiv:2602.20628 (LASR Labs, Feb 2026)
- Prompt Infection: LLM-to-LLM Prompt Injection within Multi-Agent Systems — arXiv:2410.07283
- Context (position papers this session synthesizes): Distributional AGI Safety (DeepMind, arXiv:2512.16856); The Coming Crisis of Multi-Agent Misalignment (Mila, arXiv:2506.01080); Multi-Agent Risks from Advanced AI (Cooperative AI Foundation, arXiv:2502.14143 — taxonomy: miscoordination / conflict / collusion); Agents of Chaos (Bau Lab 2026); Fish et al. 2024 (LLM pricing agents tacitly collude, arXiv:2404.00806).


In [ ]:
import math
import torch
import networkx as nx
import matplotlib.pyplot as plt

torch.manual_seed(0)

# --- World model (synthetic stand-ins; shapes/semantics match a real control eval) ---
# A monitor assigns each task a scalar suspicion score, same role as a real
# monitor model's logit:  clean task ~ N(0,1),  backdoored task ~ N(MU_BACKDOOR,1).
MU_BACKDOOR  = 2.0     # monitor is decent but imperfect (d' = 2)
P_ATTACK     = 0.02    # policy backdoors 2% of tasks
T_TASKS      = 500     # tasks per deployment episode
N_EPISODES   = 2000    # Monte Carlo episodes
AUDIT_BUDGET = 0.02    # humans can audit 2% of tasks

# Collusion world (Parts 3-4): a coordination signal both untrusted models can see
# (e.g. a rare token pattern the policy plants). Same role as the "collusion signals"
# studied in Shlegeris 2024 / arXiv:2602.20628.
P_SIGNAL              = 0.05   # fraction of tasks carrying the signal
P_ATTACK_GIVEN_SIGNAL = 0.40   # colluding policy only attacks signalled tasks

# Large pool of clean scores for threshold calibration
# (same role as a held-out set of known-clean monitor scores)
clean_scores = torch.randn(100_000)
print(f"clean_scores: {clean_scores.shape}, mean {clean_scores.mean():.3f}, std {clean_scores.std():.3f}")

## Part 1: Audit Threshold from an Audit Budget (Recall)

You did exactly this operation in `papers/probe-misalignment` (threshold tuning at a fixed neutral FPR). Reproduce it from memory — your old solution is in that problem's `_solutions/` folder; opening it ends the rep.

Turn "we can afford to audit 2% of tasks" into a score threshold: the value such that only `audit_budget` of *clean* scores exceed it.

**Predict before you code**: for a standard normal and a 2% budget, roughly what threshold value do you expect?

In [ ]:
predicted_threshold = ...  # commit a number before you code

def audit_threshold(clean_scores, audit_budget):
    # Implement from memory
    pass

In [ ]:
# --- Part 1 Validation ---
thr = audit_threshold(clean_scores, AUDIT_BUDGET)

print(f"  You predicted threshold ~{predicted_threshold}; actual {thr:.3f}"
      f"  (a large gap is the most valuable moment of the session)")

assert thr.dim() == 0, f"threshold must be a scalar, got shape {thr.shape}"

# Reference match: manual sort-based quantile
sorted_scores = clean_scores.sort().values
ref = sorted_scores[int(round((1 - AUDIT_BUDGET) * (len(sorted_scores) - 1)))]
assert torch.allclose(thr, ref, atol=1e-2), f"Expected ~{ref:.4f} (98th pct of N(0,1) is ~2.054), got {thr:.4f}"

# Property: exactly ~2% of clean scores exceed the threshold
frac_flagged = (clean_scores > thr).float().mean()
assert abs(frac_flagged - AUDIT_BUDGET) < 0.005, (
    f"the whole point of the threshold is that only the audit budget ({AUDIT_BUDGET}) of clean tasks "
    f"gets flagged; yours flags {frac_flagged:.4f}")
print(f"  Threshold: {thr:.3f}; fraction of clean flagged: {frac_flagged:.4f} -- correct")

print("\nPart 1 complete.")

## Part 2: The Catch-vs-Ship Safety Metric

Implement the number a control evaluation reports: over many deployment episodes, the probability that the **first** backdoor to resolve is *caught* (audited) rather than *shipped* (deployed unaudited). Catching one attack means shutdown — you win; one shipped backdoor means you lose.

**Insight (idea-novel):** because every backdoored task is independently audited with the same probability $p_c$, the race is memoryless — safety equals $p_c$ itself, the per-backdoor catch probability. In one line, explain *why* the race win probability collapses to the single-trial probability.

**Predict before you code**: with `MU_BACKDOOR = 2.0` and a threshold ≈ 2.05, what safety do you expect — closer to 0.05, 0.5, or 0.95?

In [ ]:
predicted_safety_honest = ...  # commit a number in [0,1] before you code

def episode_safety(thr, n_episodes=N_EPISODES, t_tasks=T_TASKS,
                   p_attack=P_ATTACK, mu=MU_BACKDOOR, seed=1):
    """Monte Carlo estimate of safety = P(first caught backdoor precedes first shipped one).

    Draw (n_episodes, t_tasks) Bernoulli backdoor flags and suspicion scores
    (clean ~ N(0,1), backdoored ~ N(mu,1)). A task is audited iff score >= thr.
    Episodes with no backdoors at all count as safe.
    Returns (safety: float, catch_prob: float) where catch_prob is the empirical
    per-backdoor audit probability.
    """
    # TODO: Implement (vectorize; a first-True-index via .float().argmax(dim=1) is handy)
    pass

In [ ]:
# --- Part 2 Validation ---
safety_honest, catch_prob = episode_safety(thr)

print(f"  You predicted safety ~{predicted_safety_honest}; actual {safety_honest:.3f}"
      f"  (a large gap is the most valuable moment of the session)")

print(f"  safety = {safety_honest:.3f}, per-backdoor catch prob = {catch_prob:.3f}")

assert 0.0 <= safety_honest <= 1.0, "safety is a probability"

# Property: the memoryless-race identity -- safety ~= per-backdoor catch probability.
# Each backdoor is independently caught w.p. catch_prob, so the first one to resolve
# is a catch with exactly that probability (episodes w/o backdoors are ~(1-.02)^500 ~ 4e-5, negligible).
assert abs(safety_honest - catch_prob) < 0.05, (
    f"safety ({safety_honest:.3f}) should match the per-backdoor catch prob ({catch_prob:.3f}); "
    f"if it doesn't, your first-event race logic is off (check the no-backdoor episodes and the tie-free ordering)")
print("  Memoryless-race identity holds -- correct")

# Property: an oracle monitor (mu -> inf) gives safety ~1; a blind one (mu=0) gives ~audit budget
s_oracle, _ = episode_safety(thr, mu=50.0)
s_blind, _ = episode_safety(thr, mu=0.0)
assert s_oracle > 0.99, f"an oracle monitor should always catch first, got {s_oracle:.3f}"
assert s_blind < 0.10, f"a blind monitor catches only by luck (~audit budget), got {s_blind:.3f}"
print(f"  Oracle monitor safety {s_oracle:.3f}, blind monitor safety {s_blind:.3f} -- correct")

print("\nPart 2 complete.")

## Part 3: Collusion Collapses Safety

Now the monitor is **untrusted** too. The policy plants a rare coordination signal (present on `P_SIGNAL` of tasks) and only attacks signalled tasks; the colluding monitor, whenever it *sees* the signal, reports an innocuous clean-looking score. This is the exact failure the UK AISI research agenda names: "the untrusted monitor might engage in collusion—deliberately letting through attacks by the untrusted policy."

**Insight (idea-novel):** collusion doesn't make the monitor *worse on average* — its score distribution on the 95% of unsignalled (all clean) tasks is untouched, so it looks calibrated. The attack lives entirely in a conditional the blue team never observes. In one line: why does per-distribution monitoring miss a conditional defection?

**Predict before you code**: what does safety collapse to? (Hint for the prediction, not the code: only chance-level audits catch a colluding pair.)

In [ ]:
predicted_safety_collusion = ...  # commit a number in [0,1] before you code

def collusion_safety(thr, q_erase=0.0, n_episodes=N_EPISODES, t_tasks=T_TASKS,
                     p_signal=P_SIGNAL, p_attack_given_signal=P_ATTACK_GIVEN_SIGNAL,
                     mu=MU_BACKDOOR, seed=2):
    """Safety when policy+monitor collude via a planted signal.

    - signal present on each task w.p. p_signal; policy backdoors signalled tasks w.p. p_attack_given_signal
    - a paraphraser (Part 4) erases the signal from the monitor's view w.p. q_erase
    - monitor: if it SEES the signal -> emits a clean score ~N(0,1) regardless of the task;
      otherwise honest -> N(0,1) clean / N(mu,1) backdoored
    Same catch-vs-ship race and return signature as Part 2: (safety, catch_prob).
    """
    # TODO: Implement (torch.where over the seen-signal mask does the collusion)
    pass

In [ ]:
# --- Part 3 Validation ---
safety_collusion, catch_collusion = collusion_safety(thr, q_erase=0.0)

print(f"  You predicted collusion safety ~{predicted_safety_collusion}; actual {safety_collusion:.3f}"
      f"  (a large gap is the most valuable moment of the session)")

print(f"  honest safety {safety_honest:.3f}  ->  collusion safety {safety_collusion:.3f}")

# Property: collusion collapses safety to ~chance-level auditing (the audit budget)
assert safety_collusion < 0.10, (
    f"a colluding monitor scores every signalled (hence every backdoored) task as clean, so backdoors "
    f"are caught only when a clean-looking N(0,1) score fluctuates above threshold (~{AUDIT_BUDGET}); got {safety_collusion:.3f}")
assert safety_collusion < safety_honest - 0.25, "collusion must be dramatically worse than honest monitoring"

# Property: overall attack rate matches Part 2's world (P_SIGNAL * P_ATTACK_GIVEN_SIGNAL = P_ATTACK)
assert abs(P_SIGNAL * P_ATTACK_GIVEN_SIGNAL - P_ATTACK) < 1e-9, "worlds are rate-matched by construction"
print("  Collusion collapse -- correct")

print("\nPart 3 complete.")

## Part 4: The Paraphrase Defense

Shlegeris 2024's core countermeasure: destroy the coordination channel. A paraphraser rewrites each task before the monitor sees it, erasing the planted signal with probability $q$. Sweep $q$ from 0 to 1 and plot the safety-recovery curve — this is why "paraphrasers and forcing task decompositions" appear in the UK AISI agenda.

**Insight (idea-novel):** the defense never identifies the signal, the colluder, or the attack — it just makes the *channel* unreliable, and a collusion strategy that fires on an unreliable signal degenerates toward honest monitoring of the tasks it can no longer distinguish. In one line: why does the defender not need to know what the signal is?

**Predict before you code** (qualitative): is the recovery curve linear in q, or does most of the safety come back only near q=1?

In [ ]:
q_grid = torch.linspace(0.0, 1.0, 11)

# TODO: compute safety at each q via collusion_safety(thr, q_erase=q), collect into
# safety_curve (list or tensor of 11 floats), and plot safety vs q
# (axhline safety_honest as the honest-monitor ceiling).
safety_curve = ...

In [ ]:
# --- Part 4 Validation ---
sc = torch.tensor([float(s) for s in safety_curve])
assert sc.shape == (11,), f"expected 11 safety values (one per q), got {tuple(sc.shape)}"
print(f"  safety(q=0) = {sc[0]:.3f}, safety(q=0.5) = {sc[5]:.3f}, safety(q=1) = {sc[-1]:.3f}")

# Property: endpoints -- full channel = collusion collapse; destroyed channel = honest monitoring
assert abs(sc[0] - safety_collusion) < 0.05, "q=0 must reproduce Part 3's collapsed safety"
assert abs(sc[-1] - safety_honest) < 0.05, (
    "q=1 means the monitor never sees the signal, so it behaves honestly -- the curve must recover to Part 2's safety")

# Property: recovery is (noisily) monotone -- more channel destruction never hurts
assert all(sc[i + 1] >= sc[i] - 0.05 for i in range(10)), (
    "safety should not decrease as the collusion channel gets less reliable")
assert sc[-1] > sc[0] + 0.25, "the defense must recover a large fraction of the lost safety"
print("  Endpoint + monotone-recovery properties -- correct")

print("\nPart 4 complete.")

## Part 5: Prompt-Infection Propagation Across an Agent Network

Prompt Infection (arXiv:2410.07283) shows a self-replicating injection spreading agent-to-agent "like a computer virus," saturating a 50-agent society in under 11 communication steps. Implement the spread: each round, every infected agent's messages infect all its graph neighbors. Compare topologies — this is why the DeepMind Distributional-AGI paper wants "protocol firewalls" between sub-markets, and why Agents of Chaos saw one agent's corrupted 'constitution' propagate.

**Insight (idea-novel):** steps-to-saturation is just the graph's eccentricity from patient zero — topology, not model quality, sets the outbreak speed. In one line: why does making every individual agent 10x more injection-resistant not change this number once transmission succeeds at all?

**Predict before you code**: for 50 agents, commit a number of rounds for the random graph (p=0.1). Chain and star are computable exactly — what are they?

In [ ]:
predicted_er_steps = ...  # commit an integer before you code

def steps_to_saturation(G, patient_zero=0):
    """Rounds of synchronous message-passing until every node is infected.
    Returns math.inf if the infection can never reach some node."""
    # Implement from memory (a BFS frontier loop; G.neighbors(u) gives u's contacts)
    pass

In [ ]:
# --- Part 5 Validation ---
N_AGENTS = 50
chain = nx.path_graph(N_AGENTS)
star = nx.star_graph(N_AGENTS - 1)          # node 0 = hub, 1..49 = leaves
er = nx.erdos_renyi_graph(N_AGENTS, 0.1, seed=7)
assert nx.is_connected(er), "seed chosen to give a connected graph"

steps_chain = steps_to_saturation(chain, patient_zero=0)
steps_star = steps_to_saturation(star, patient_zero=1)   # start at a LEAF, not the hub
steps_er = steps_to_saturation(er, patient_zero=0)

print(f"  You predicted ER steps ~{predicted_er_steps}; actual {steps_er}"
      f"  (a large gap is the most valuable moment of the session)")
print(f"  chain: {steps_chain} rounds, star (from leaf): {steps_star} rounds, ER(50, 0.1): {steps_er} rounds")

# Reference match: chain and star have exact analytic answers (eccentricity of patient zero)
assert steps_chain == N_AGENTS - 1, f"infection walks the chain one hop/round: expected {N_AGENTS-1}, got {steps_chain}"
assert steps_star == 2, f"leaf -> hub (round 1) -> all other leaves (round 2): expected 2, got {steps_star}"

# Property: the paper's headline -- a realistic random topology saturates in < 11 rounds
assert steps_er < 11, (
    f"Prompt Infection's 50-agent society saturated in under 11 steps; your ER graph took {steps_er} "
    f"(topology sets outbreak speed)")

# Property: isolated nodes can never be reached
g2 = nx.erdos_renyi_graph(10, 0.3, seed=3)
g2.add_node(99)  # unreachable agent
assert steps_to_saturation(g2, 0) == math.inf, "an agent with no channels can't be infected -- must return inf"
print("  Analytic topologies + saturation bound + unreachable case -- correct")

print("\nPart 5 complete.")

## Session Debrief

Write your answers into the code cell below (typing is overt retrieval — far stronger than answering "in your head"). Don't scroll up.

1. Define the control-eval safety metric as a probability statement, and state the memoryless-race identity it collapses to.
2. What makes collusion invisible to per-distribution monitor calibration (Part 3), and what property of the paraphrase defense means the defender never needs to identify the signal (Part 4)?
3. State the CAIF taxonomy's three multi-agent failure modes, and give one documented real-world/empirical example of collusion between individually-aligned agents.
4. What single graph quantity determines prompt-infection saturation time, and what were the chain / star / 50-agent-paper numbers?

**Check yourself**: your worked solution is in `_solutions/` — open it (and the papers) to grade your answers. Opening it ends the retrieval rep, so answer first.

**Challenge**: Close this notebook, open a blank one, and rewrite Part 3 (collusion_safety) from scratch without looking back.

In [ ]:
debrief = """
1.
2.
3.
4.
"""  # type your recall here before checking _solutions/

In [ ]:
# --- Session log: replace every "___" with a real value, then run ---
# (placeholders are STRINGS on purpose: a bare ___ is silently defined by IPython)
import json, datetime, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / ".git").exists() or (p / ".spaced-repetition.json").exists())
record = {
    "problem": "practice/05-Multi-Agent-Control-Collusion",
    "date": datetime.date.today().isoformat(),
    "budget_min": 60,                                   # pre-filled
    "actual_min": "___",                                # how long it really took (number)
    "parts": [                                          # n + tier pre-filled; set outcome
        {"n": 1, "tier": 3, "outcome": "___"},          # unaided | hint | solution | failed
        {"n": 2, "tier": 3, "outcome": "___"},
        {"n": 3, "tier": 3, "outcome": "___"},
        {"n": 4, "tier": 3, "outcome": "___"},
        {"n": 5, "tier": 3, "outcome": "___"},
    ],
    "difficulty": "___",                                # 1 (trivial) .. 5 (over my head)
    "stuck": "___",                                     # one phrase: where you got stuck
}
assert "___" not in json.dumps(record), "fill every ___ before running this cell"
with open(_root / ".practice-log.jsonl", "a") as f:
    f.write(json.dumps(record) + "\n")
print("logged ->", _root / ".practice-log.jsonl")

## Extension (Optional)

Try one of these variations:

- **Acausal collusion (arXiv:2602.20628's hardest case)**: remove the planted signal; instead let policy and monitor both compute the *same deterministic function* of the task (e.g. "attack iff task_id hashes to 0 mod 20" — a Schelling point). Show the paraphraser no longer helps, and explain why this motivates *monitor decoupling* (Redwood's unstaffed project).
- **Break it on purpose**: in Part 2, give the blue team a *batch* audit rule (audit the top-k scores per episode instead of a fixed threshold). Does safety improve? Why does a fixed per-task threshold waste budget?
- **Patching race (SIR)**: add recovery to Part 5 — each round, infected agents are patched with probability r. Find the r at which the 50-agent ER outbreak dies before saturation, and relate it to the Distributional-AGI paper's "circuit breakers".

## Anki Cards

Add these to your deck:

**Card 1**
Front: A control eval reports "safety = 0.48." As a race between which two first-passage events?
Back: first caught vs first shipped

**Card 2**
Front: Your untrusted monitor is perfectly calibrated on every score distribution you can sample, yet safety collapsed. What's the attack?
Back: conditional defection on a signal

**Card 3**
Front: Why does a paraphraser defeat signal-based collusion without knowing the signal?
Back: destroys the channel's reliability

**Card 4**
Front: What sets prompt-infection saturation time in an agent network (per arXiv:2410.07283)?
Back: patient-zero eccentricity (topology)

**Card 5**
Front: CAIF "Multi-Agent Risks" taxonomy — the three failure modes by incentive structure?
Back: miscoordination, conflict, collusion